# 🧠 深度学习进阶：文字接龙与向量空间的奥秘

湖北理工学院《人工智能理论》课程资料

📝 **本节内容概述：**
我们将深入探索大语言模型（LLM）的核心任务——**文字接龙（Next Token Prediction）**。通过对比从基础到企业级的多种实现方案，理解词法分析与语义表示的演进。

1. 📊 **数据准备**：使用完整的自定义语料库进行训练。
2. 🧩 **方案一：基础版**：字符级分词 + 3 层深度网络，理解最原始的概率预测。
3. ⚡ **方案二：进阶版**：子词级分词 (BPE) + 分步训练嵌入层，掌握现代大模型的标配方案。
4. 🏢 **方案三：企业版**：现成的分词器与预训练嵌入，体验迁移学习的力量。
5. 📉 **对比实验**：分析上下文窗口大小对预测准确率的影响。

> 💡 **核心理念**：大模型并不神秘。从一个字到万亿参数，其底层逻辑始终是：**给定上下文，预测下一个最可能的符号**。

## 📊 1. 数据准备

我们加载 `./data/custom_pretrain_corpus.txt` 中的所有语料。为了确保实验质量，我们仅保留纯中文句子，并去除重复项。

In [ ]:
import re
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import clear_output
from sklearn.metrics import accuracy_score
import tiktoken

# ========================================================
# 👇 设置中文字体，确保图表正常显示汉字
# ========================================================
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

# ========================================================
# 👇 加载并清洗语料
# ========================================================
with open('./data/custom_pretrain_corpus.txt', 'r', encoding='utf-8') as f:
    all_lines = [line.strip() for line in f if line.strip()]

def is_pure_chinese(text):
    # 匹配中文字符及中文标点
    return bool(re.fullmatch(r'[\u4e00-\u9fff\u3000-\u303f\uff00-\uffef，。！？、；：""\'\'（）《》【】]+', text))

corpus = [line for line in all_lines if is_pure_chinese(line)]
corpus = list(dict.fromkeys(corpus)) # 去重

print(f'✅ 语料加载完成！共 {len(all_lines)} 行，去重筛选后剩余 {len(corpus)} 条纯中文语料。')
print(f'📜 样例展示：{corpus[:3]}')

## 🧩 方案一：基础文字接龙 (字符级 + 3 层 FCN)

### 构建简单的字符级分词器 (Tokenizer) 
计算机无法直接理解汉字。**分词 (Tokenization)** 的任务就是将连续的文本切分成离散的单元（Token）。

在最基础的方案中，我们使用**逐字切分**。其优势在于模型结构简单，词汇表（Vocabulary）大小可控（约几千个汉字即可覆盖常用表达）。为了让模型知道一句话的开始、结束或填充位置，我们引入了特殊标记：
- `<PAD>` (0): 填充符，用于对齐不等长的句子，让计算机能进行矩阵并行计算。
- `<EOS>` (1): 结束符，标志一句话的终点，防止模型生成到停不下来。

In [ ]:
# 1. 构建字符级词表
all_chars = "".join(corpus)
unique_chars = sorted(list(set(all_chars)))

# 约定特殊标记：<PAD> 为 0（用于填充），<EOS> 为 1（用于标识句子结束）。
# 这种“特殊符号”设计是大语言模型（LLM）处理变长序列的基础。
vocab_v1 = ["<PAD>", "<EOS>"] + unique_chars

# char2idx: 汉字 -> 数字编号 (用于计算机运算)
char2idx = {char: i for i, char in enumerate(vocab_v1)}
# idx2char: 数字编号 -> 汉字 (用于将模型预测结果转换回人类可读文字)
idx2char = {i: char for i, char in enumerate(vocab_v1)}
vocab_size_v1 = len(vocab_v1)

print(f"📌 字符级词表构建完成！词数: {vocab_size_v1}")

# 2. 将文本语料转换为数字序列
def text_to_seq_v1(text):
    # 逐字转换，末尾统一添加 <EOS> 标记
    return [char2idx[c] for c in text] + [char2idx["<EOS>"]]

corpus_seqs_v1 = [text_to_seq_v1(line) for line in corpus]

### 嵌入 (Embedding)及样本生成

在本方案中，我们直接使用字符的 **ID (索引)** 作为特征输入。这是一种最原始的“硬编码”方式。模型需要通过全连接层的权重，从 1 维数字中通过数学变化扩展到高维，尝试理解这些数字背后代表的语义规律。

In [ ]:
# 3. 滑动窗口构造样本
# window_size = 4 表示“用前4个字预测第5个字”，这是模型感知上文的“视野”范围。
window_size = 4
X_data_v1, y_data_v1 = [], []

for seq in corpus_seqs_v1:
    if len(seq) <= window_size: continue
    # 在句子上逐字滑动
    for i in range(len(seq) - window_size):
        # 取前 4 个字作为输入特征 X
        X_data_v1.append(seq[i : i + window_size])
        # 取紧随其后的第 5 个字作为预测目标 y
        y_data_v1.append(seq[i + window_size])

# 转换为 PyTorch 张量
# X_tensor 设为 float32 以便直接输入全连接层计算
X_tensor_v1 = torch.tensor(X_data_v1, dtype=torch.float32)
# y_tensor 设为 long，因为分类问题的标签必须是整数索引
y_tensor_v1 = torch.tensor(y_data_v1, dtype=torch.long)

print(f"📊 样本构建完成！总样本数: {len(X_tensor_v1)}")

### 模型构建及训练：3 层隐藏层 FCN
为了拟合复杂的语言规律，我们不再使用单层回归模型。我们构建一个拥有 **3 个隐藏层** 的全连接神经网络，配合 **ReLU** 激活函数，增加模型“转弯”的能力。

In [ ]:
class BasicWordChainModel(nn.Module):
    def __init__(self, vocab_size, window_size):
        super().__init__()
        # 定义 3 层隐藏层，通过增加网络的深度来捕捉更复杂的语言逻辑关系
        self.net = nn.Sequential(
            nn.Linear(window_size, 128), # 输入层
            nn.ReLU(),                   # 激活函数：让线性模型学会“拐弯”
            nn.Linear(128, 128),         # 隐藏层 1
            nn.ReLU(),
            nn.Linear(128, 128),         # 隐藏层 2
            nn.ReLU(),
            nn.Linear(128, vocab_size)   # 输出层：映射到词表大小
        )
    
    def forward(self, x):
        return self.net(x)

# 初始化模型
model_v1 = BasicWordChainModel(vocab_size_v1, window_size)
optimizer_v1 = optim.Adam(model_v1.parameters(), lr=0.005) # Adam: 优化器
criterion = nn.CrossEntropyLoss() # 损失函数

epochs = 1200
loss_history_v1 = []
num_samples = X_tensor_v1.shape[0]

print("🚀 方案一启动训练（含随机打乱）...")
for epoch in range(epochs):
    # ========================================================
    # 🎲 关键步骤：随机打乱样本顺序
    # ========================================================
    indices = torch.randperm(num_samples) # 生成随机索引序列
    X_shuffled = X_tensor_v1[indices]
    y_shuffled = y_tensor_v1[indices]
    
    # 前向传播：计算预测结果
    logits = model_v1(X_shuffled)
    # 计算损失：衡量预测结果与打乱后真实标签的偏差
    loss = criterion(logits, y_shuffled)
    
    # 反向传播与权重更新
    optimizer_v1.zero_grad() 
    loss.backward()          
    optimizer_v1.step()      
    
    loss_history_v1.append(loss.item())
    
    # 每 200 轮可视化一次
    if (epoch + 1) % 200 == 0 or epoch == 0:
        clear_output(wait=True)
        plt.figure(figsize=(10, 4))
        plt.plot(loss_history_v1, color='dodgerblue', label='Train Loss')
        plt.title(f"方案一：训练阶段 (Epoch {epoch+1}, Shuffled, 3-Layer FCN)")
        plt.xlabel("Epoch")
        plt.ylabel("Loss")
        plt.grid(alpha=0.3)
        plt.legend()
        plt.show()
        
print(f"✅ 方案一训练完成！最终 Loss: {loss_history_v1[-1]:.4f}")

### 方案一结果测试：自回归生成
我们将训练好的模型进行“文字接龙”实测。给定一个起始词（Prompt），由模型自主预测后续内容，并将其“喂回”给模型作为新的输入，这种过程称为**自回归生成**。

In [ ]:
def generate_text_v1(start_text, model, max_len=50):
    model.eval()
    # 将初始文字转为索引 ID 序列
    current_seq = [char2idx[c] for c in start_text]
    result = list(start_text)
    
    for _ in range(max_len):
        # 始终取最后的 window_size 个字作为输入
        input_temp = current_seq[-window_size:]
        input_x = torch.tensor([input_temp], dtype=torch.float32)
        
        with torch.no_grad():
            logits = model(input_x)
            # 获取概率最高的一个字的索引
            next_idx = torch.argmax(logits, dim=1).item()
        
        # 将新字加入结果序列和当前输入缓冲区
        result.append(idx2char[next_idx])
        current_seq.append(next_idx)
        
        # 遇到结束符则停止生成
        if idx2char[next_idx] == "<EOS>": break
        
    return "".join(result).replace("<EOS>", "")

# --- 测试结果 1 ---
prompt = "模型训练"
output = generate_text_v1(prompt, model_v1)
print(f"💡 输入（Prompt）1：{prompt}")
print(f"✍️ 生成结果：{output}")

# --- 测试结果 2 ---
prompt = "模型训练"
output = generate_text_v1(prompt, model_v1)
print(f"💡 输入（Prompt）2：{prompt}")
print(f"✍️ 生成结果：{output}")

## ⚡ 方案二：进阶版 (BPE 分词 + 分步训练嵌入层)

### Tokenizer 分词：BPE (Byte Pair Encoding)方法
字符级分词虽然简单，但效率较低（如“机器学习”需 4 个 Token）。**BPE** 是现代大模型（如 GPT-4）的标配，它能自动识别并合并高频词组。
#### 🔍 为什么 BPE 的数字序列（ID）会很大？
当你运行 BPE 时，你会发现 Token 的编号往往高达 **100,000** 以上，这是因为：
1. **全球通用大字典**：我们使用的是 `tiktoken` 的 `cl100k_base` 编码，它预置了一个包含约 10 万个 Token 的巨型词表，涵盖了全球多语言、代码和特殊符号。
2. **绝对位置映射**：即使我们的语料库很小，每个汉字或词组在 `tiktoken` 的这本“大字典”里都有一个固定的绝对编号。
3. **子词合并深度**：BPE 通过合并极其丰富的字符组合，产生了一套极其庞大的符号系统。
> 💡 **核心策略**：为了让神经网络计算更高效，我们并不直接使用这些巨大的原始 ID，而是建立一个 **“局部映射表” (Local Mapping)**，将它们重新映射回我们语料库涉及的、更小的连续索引范围（如 0~3000）。

In [ ]:
# 1. 加载 GPT-4 字典
enc = tiktoken.get_encoding("cl100k_base")

# 2. 编码语料，并手动在句末添加一个全局字典中不存在的特殊 ID 作为 EOS (比如 100000)
MAGIC_EOS_ID = 100000
corpus_seqs_v2_raw = [enc.encode(line) + [MAGIC_EOS_ID] for line in corpus]

# 📝 建立局部映射表 (Local Mapping)
# 为什么要映射？全局字典有 10 万个 ID，直接训练会导致 Embedding 矩阵巨大。
# 我们只提取当前语料出现过的 Token，将其映射到 0, 1, 2... 的连续小编号。
all_bpe_tokens = [t for seq in corpus_seqs_v2_raw for t in seq]
unique_bpe = sorted(list(set(all_bpe_tokens)))

bpe2idx = {t: i for i, t in enumerate(unique_bpe)}
idx2bpe = {i: t for i, t in enumerate(unique_bpe)}
vocab_size_v2 = len(unique_bpe)

# 转换序列到局部小编号空间 (0 ~ vocab_size_v2)
corpus_seqs_v2_mapped = [[bpe2idx[t] for t in seq] for seq in corpus_seqs_v2_raw]

print(f"📌 局部词表构建完成！已手动加入 EOS 停止信号。")
print(f"🔢 局部 ID {bpe2idx[MAGIC_EOS_ID]} 现在代表句子结束。")

### 词嵌入（Embedding）：向量空间中的语义搜索
**嵌入 (Embedding)** 的本质是建立一个查表矩阵：`[词表大小, 向量维度]`。它将离散的数字 ID 映射到高维连续向量空间。
- 在空间中，语义相似的词（如“机器”和“深度”）对应的向量距离会更近。

### 🚜 练习：分步训练 (Separate Training)
我们将训练分为两个阶段，以深刻体验 Embedding 的作用：
1. **第一阶段：预训练 Embedding**。利用 Bigram 模型让 Embedding 层学习到基础语义。
2. **第二阶段：固定 Embedding 训练 FCN**。冻结 Embedding 权重的更新，仅训练后续的 3 层全连接层。

In [ ]:
# 设定嵌入维度为 64（比基础版更高，以容纳更复杂的语义）
embed_dim = 64
emb_layer = nn.Embedding(vocab_size_v2, embed_dim)

# --- 步骤 A: 准备 Bigram (二元) 训练数据 ---
# Bigram 的目标是：给定当前词，预测下一个词。这能强迫 Embedding 层学习到词与词的关联。
bigram_X, bigram_y = [], []
for seq in corpus_seqs_v2_mapped:
    for i in range(len(seq) - 1):
        bigram_X.append(seq[i])
        bigram_y.append(seq[i+1])

bigram_X_t = torch.tensor(bigram_X, dtype=torch.long)
bigram_y_t = torch.tensor(bigram_y, dtype=torch.long)

# --- 步骤 B: 定义简单的预训练模型 ---
class BigramPreTrainer(nn.Module):
    def __init__(self, emb, out_dim):
        super().__init__()
        self.embedding = emb # 共享我们要训练的 Embedding 层
        self.linear = nn.Linear(embed_dim, out_dim)
        
    def forward(self, x): 
        # 输入 ID -> 查表得到向量 -> 线性层映射到词表概率
        return self.linear(self.embedding(x))

pre_trainer = BigramPreTrainer(emb_layer, vocab_size_v2)
pre_opt = optim.Adam(pre_trainer.parameters(), lr=0.01)

# --- 步骤 C: 执行预训练 ---
print("⏳ 正在预训练 Embedding 层 (通过预测下一个 Token 学习语义)...")
for epoch in range(500):
    logits = pre_trainer(bigram_X_t)
    loss = criterion(logits, bigram_y_t)
    
    pre_opt.zero_grad()
    loss.backward()
    pre_opt.step()
    
    if (epoch + 1) % 100 == 0:
        print(f"Epoch {epoch+1:3d} | Loss: {loss:.4f}")

print("✅ Embedding 预训练完成！")

在二维空间中展示 Embedding 结果

In [ ]:
from sklearn.decomposition import PCA
import re

# 获取 Embedding 权重
word_vectors = emb_layer.weight.detach().numpy()

# PCA 降维至 2D
pca = PCA(n_components=2)
word_vectors_2d = pca.fit_transform(word_vectors)

plt.figure(figsize=(10, 8))
plt.scatter(word_vectors_2d[:, 0], word_vectors_2d[:, 1], c='darkorange', alpha=0.5)

# ✅ 优化：通过正则过滤，仅展示纯汉字 Token
def is_chinese(text):
    return bool(re.fullmatch(r'[\u4e00-\u9fff]+', text))

for i in range(vocab_size_v2):
    raw_id = idx2bpe[i]
    if raw_id == MAGIC_EOS_ID:
        token_str = "<EOS>" # 展示手动加入的结束符
    else:
        # 解开 BPE 编码查看其真面目
        token_str = enc.decode([raw_id])
    
    # 💡 只有是汉字或者 EOS 时，我们才在图上打标签，避免乱码干扰
    if is_chinese(token_str) or token_str == "<EOS>":
        plt.annotate(token_str, (word_vectors_2d[i, 0], word_vectors_2d[i, 1]), 
                     fontsize=12, family='SimHei', weight='bold')

plt.title("方案二：预训练 Embedding 结果 (仅展示汉字 Token)")
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# --- 第二阶段：训练深度分类器 ---
# 冻结权重
emb_layer.weight.requires_grad = False 

X_data_v2, y_data_v2 = [], []
for seq in corpus_seqs_v2_mapped:
    if len(seq) <= window_size: continue
    for i in range(len(seq) - window_size):
        X_data_v2.append(seq[i : i + window_size])
        y_data_v2.append(seq[i + window_size])

X_tensor_v2 = torch.tensor(X_data_v2, dtype=torch.long)
y_tensor_v2 = torch.tensor(y_data_v2, dtype=torch.long)

class AdvWordChainModel(nn.Module):
    def __init__(self, emb, window_size, vocab_size):
        super().__init__()
        self.embedding = emb
        self.fcn = nn.Sequential(
            nn.Linear(window_size * embed_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Linear(128, vocab_size)
        )
    
    def forward(self, x):
        # x: [batch, window_size] -> emb: [batch, window_size, dim] -> flat: [batch, window_size*dim]
        e = self.embedding(x).view(x.size(0), -1)
        return self.fcn(e)

model_v2 = AdvWordChainModel(emb_layer, window_size, vocab_size_v2)
optimizer_v2 = optim.Adam(model_v2.fcn.parameters(), lr=0.002)

loss_history_v2 = []
for epoch in range(1200):
    logits = model_v2(X_tensor_v2)
    loss = criterion(logits, y_tensor_v2)
    optimizer_v2.zero_grad()
    loss.backward()
    optimizer_v2.step()
    
    loss_history_v2.append(loss.item())
    if (epoch + 1) % 200 == 0:
        clear_output(wait=True)
        plt.figure(figsize=(10, 4))
        plt.plot(loss_history_v2, color='darkorange')
        plt.title(f"方案二：BPE 分词 + 固定 Embedding 训练损失 (Epoch {epoch+1})")
        plt.show()

print(f"✅ 方案二训练完成！最终 Loss: {loss_history_v2[-1]:.4f}")

In [ ]:
def generate_text_v2_v2(start_text, model, max_len=50):
    model.eval()
    # 1. BPE 编码
    raw_ids = enc.encode(start_text)
    # 2. 映射到局部小编号
    current_seq = [bpe2idx[t] for t in raw_ids if t in bpe2idx]
    result_ids = list(raw_ids)
    
    for _ in range(max_len):
        # 提取最后 window_size 个字作为输入
        if len(current_seq) < window_size:
            input_temp = [0] * (window_size - len(current_seq)) + current_seq
        else:
            input_temp = current_seq[-window_size:]
            
        input_x = torch.tensor([input_temp], dtype=torch.long)
        
        with torch.no_grad():
            logits = model(input_x)
            next_idx_local = torch.argmax(logits, dim=1).item()
            
            # --- 💡 关键修改：检查是否撞到了 EOS ---
            if idx2bpe[next_idx_local] == MAGIC_EOS_ID:
                print("🛑 模型预测到了停止符，接龙结束。")
                break
                
            next_token_id = idx2bpe[next_idx_local]
        
        # 将新 Token 加入序列，继续下一轮循环
        result_ids.append(next_token_id)
        current_seq.append(next_idx_local)
        
    return enc.decode(result_ids)

# --- 实测 ---
prompt = "模型训练"
print(f"💡 输入 1：{prompt}")
print(f"✍️ 生成结果：{generate_text_v2_v2(prompt, model_v2)}")


# --- 实测 ---
prompt = "模型训练"
print(f"💡 输入 2：{prompt}")
print(f"✍️ 生成结果：{generate_text_v2_v2(prompt, model_v2)}")

## 🏢 方案三：企业级方案 (现成 Tokenizer + 现成 Embedding)

在实际工业应用中，我们往往不需要从头训练词向量。我们可以直接通过 API 或企业知识库提供的 **Embedder** 获取高质量的特征向量。这极大地利用了大规模语料的先验知识。

In [ ]:
import os
from openai import OpenAI
import torch
import torch.nn as nn

# --- 1. 配置硅基流动客户端 ---
# ============================================================
# 👇 把你的 SiliconFlow API Key 填在下面的引号里
# ============================================================
SILICONFLOW_API_KEY = ""  # 例如: "sk-abc123..."

client = OpenAI(
    api_key=SILICONFLOW_API_KEY, 
    base_url="https://api.siliconflow.cn/v1", # 硅基流动的 API 终端地址
)

class SiliconFlowEmbedder:
    """具备自动分批能力的硅基流动 BGE 嵌入模型调用器"""
    def __init__(self, model="BAAI/bge-large-zh-v1.5"):
        self.model = model
        self.dimension = 1024
        self.max_batch_size = 32 # 🛡 硅基流动的安全阈值
        
    def embed_batch(self, texts):
        """支持自动切分长列表的分批调用"""
        all_vectors = []
        
        # 将输入列表按 32 个为一组进行切割循环
        for i in range(0, len(texts), self.max_batch_size):
            batch_texts = texts[i : i + self.max_batch_size]
            try:
                response = client.embeddings.create(
                    model=self.model,
                    input=batch_texts
                )
                # 记录本批次的向量
                batch_vectors = [item.embedding for item in response.data]
                all_vectors.extend(batch_vectors)
            except Exception as e:
                print(f"❌ 硅基流动分批调用失败 (索引 {i}): {e}")
                # 容错：如果某批失败，用随机向量填充该批次
                all_vectors.extend([np.random.randn(self.dimension).tolist()] * len(batch_texts))
        
        return torch.tensor(all_vectors, dtype=torch.float32)

# --- 重新测试离线预取 ---
enterprise_api = SiliconFlowEmbedder()
# 此时再运行之前的方案三训练单元格，309 个词将会分为 10 个批次自动完成。
print(f"✅ 硅基流动 Embedder 已连接（当前模型：{enterprise_api.model}，维度：1024）")
print(f"✅ 单次请求上限：{enterprise_api.max_batch_size}")

我们从 custom_pretrain_corpus.txt 语料库中提取核心术语。

这些词汇在语料中频繁出现，属于深度学习的核心概念。我们将通过 硅基流动 (BGE) 看看它们在 1024 维空间中是如何自动排布的。

In [ ]:
import numpy as np
from sklearn.decomposition import PCA

# --- 1. 从语料库中精选核心术语 ---
corpus_keywords = [
    # 模型架构类
    "深度学习", "机器学习", "神经网络", "Transformer", "自注意力",
    # 训练机制类
    "反向传播", "梯度下降", "学习率", "损失", "参数",
    # 特征处理类
    "词向量", "Tokenizer", "Embedding", "位置编码",
    # 开发部署类
    "预训练", "微调", "GPU", "上下文窗口", "微调"
]

# --- 2. 获取硅基流动 BGE 高维特征 ---
print(f"🚀 正在为语料库中的 {len(corpus_keywords)} 个核心术语请求语义特征...")
corpus_vectors = enterprise_api.embed_batch(corpus_keywords).numpy()

# --- 3. PCA 降维 ---
pca_engine = PCA(n_components=2)
coords = pca_engine.fit_transform(corpus_vectors)

# --- 4. 可视化绘图 ---
plt.figure(figsize=(14, 11))
# 绘制散点
plt.scatter(coords[:, 0], coords[:, 1], c='crimson', edgecolors='white', s=120, zorder=3)

# 标注术语
for i, txt in enumerate(corpus_keywords):
    plt.annotate(txt, (coords[i, 0], coords[i, 1]), 
                 xytext=(5, 5), textcoords='offset points',
                 fontsize=13, family='SimHei', weight='bold',
                 bbox=dict(boxstyle='round,pad=0.2', fc='wheat', alpha=0.6))

plt.title("方案三：语料库核心术语的语义拓扑图 (By SiliconFlow BGE)", fontsize=16)
plt.xlabel("主成分 1 (PC1)", fontsize=12)
plt.ylabel("主成分 2 (PC2)", fontsize=12)
plt.grid(True, linestyle=':', alpha=0.5)
plt.show()

# --- 💡 教学解读 ---
print("🎓 扫码观察：")
print("- 聚类现象：你会发现'机器学习'、'深度学习'、'神经网络'由于经常出现在优化的上下文中，自然会聚在一起。")
print("- 架构聚合：'Transformer'、'词向量'、'位置编码'会由于属于同一技术体系而靠得很近。")

训练文字接龙模型

In [ ]:
# ⏳ 步骤 1：离线预取 (此时会调用一次 API 批量处理 309 个词)
# 做完这一步后，我们就再也不需要访问硅基流动了
print("⏳ 正在离线预取 BGE 词向量 (共 " + str(vocab_size_v2) + " 个)...")
all_token_texts = [enc.decode([idx2bpe[i]]) for i in range(vocab_size_v2)]
fixed_embeddings_v3 = enterprise_api.embed_batch(all_token_texts) # [309, 1024]

# ⏳ 步骤 2：组装训练张量 (这一步将“文字接龙”逻辑转为“矩阵运算”逻辑)
X_list_v3, y_list_v3 = [], []
for seq in corpus_seqs_v2_mapped:
    if len(seq) <= 4: continue
    for i in range(len(seq) - 4):
        # 从内存中查表获取 4 个向量，并拼成 4096 维
        combined = fixed_embeddings_v3[seq[i : i+4]].view(-1)
        X_list_v3.append(combined)
        y_list_v3.append(seq[i+4])

# 将列表转换为整块的内存张量 (这是提速的关键！)
X_train_v3 = torch.stack(X_list_v3) # [样本数, 4096]
y_train_v3 = torch.tensor(y_list_v3, dtype=torch.long) # [样本数]

print(f"✅ 内存张量构建完成：输入形状 {X_train_v3.shape}")

# 3. 定义模型 (保持 4096 维输入)
class FullFeatureFCN(nn.Module):
    def __init__(self, in_features, vocab_size):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_features, 128), # 输入层
            nn.ReLU(),                   # 激活函数：让线性模型学会“拐弯”
            nn.Linear(128, 128),         # 隐藏层 1
            nn.ReLU(),
            nn.Linear(128, 128),         # 隐藏层 2
            nn.ReLU(),
            nn.Linear(128, vocab_size)   # 输出层：映射到词表大小
        )
    def forward(self, x): return self.net(x)

model_v3 = FullFeatureFCN(4096, vocab_size_v2)
optimizer_v3 = optim.Adam(model_v3.parameters(), lr=0.001)

# 4. 本地训练 (没有任何 API 调用)
epochs_v3 = 1000
loss_history_v3 = []
print("🚀 训练启动...")
for epoch in range(epochs_v3):
    logits = model_v3(X_train_v3)
    loss = criterion(logits, y_train_v3)
    
    optimizer_v3.zero_grad()
    loss.backward()
    optimizer_v3.step()
    
    loss_history_v3.append(loss.item())
    
    # 每 100 轮刷新一次图像
    if (epoch + 1) % 100 == 0 or epoch == 0:
        clear_output(wait=True)
        plt.figure(figsize=(10, 4))
        plt.plot(loss_history_v3, color='crimson', label='Full Feature Loss (4096D)')
        plt.title(f"方案三训练路线：BGE 官方嵌入 (Epoch {epoch+1})")
        plt.xlabel("Epoch")
        plt.ylabel("Cross Entropy Loss")
        plt.grid(alpha=0.3)
        plt.legend()
        plt.show()
print(f"✅ 方案三训练完成！最终 Loss: {loss_history_v3[-1]:.6f}")

In [ ]:
def generate_text_v3_premium(prompt, model, embedder, max_len=50):
    model.eval()
    
    # 1. BPE 分词并映射为局部索引
    raw_ids = enc.encode(prompt)
    local_ids = [bpe2idx[tid] for tid in raw_ids if tid in bpe2idx]
    
    # 2. 准备实时接龙循环
    results = list(raw_ids)
    print(f"🤖 正在调用 BGE 企业级大脑进行语义对齐...")

    for _ in range(max_len):
        # 提取最后 4 个词
        if len(local_ids) < 4:
            win_ids = [0] * (4 - len(local_ids)) + local_ids
        else:
            win_ids = local_ids[-4:]
            
        # --- ⚡ 方案三的核心：特征提取 ---
        # 方式：直接从我们预计算好的 fixed_embeddings_v3 矩阵中查表
        # 注意：fixed_embeddings_v3 我们在训练前已经提前算好了 309 个词
        features = fixed_embeddings_v3[win_ids].view(1, -1) # 形状: [1, 4096]
        
        # 3. 概率预测
        with torch.no_grad():
            logits = model(features)
            next_idx_local = torch.argmax(logits, dim=1).item()
            
        # 4. 判定停止符
        if idx2bpe[next_idx_local] == MAGIC_EOS_ID:
            print("🛑 发现结束信号，生成停止。")
            break
            
        # 5. 更新状态
        local_ids.append(next_idx_local)
        results.append(idx2bpe[next_idx_local])
    
    # 6. 一次性解码输出
    return enc.decode(results)

# --- 实测环节 ---
test_prompt = "模型训练"
print(f"💡 输入：{test_prompt}")
result_v3 = generate_text_v3_premium(test_prompt, model_v3, enterprise_api)
print(f"✍️ 方案三生成的后续：\n{result_v3}")

“在小型‘实验室’语料（Tiny Data）下，自建词表和字符级分词往往比昂贵的预训练 Embedding 更具优势；而当数据量上升到 PB 级时，预训练 Embedding 的常识能力才会成为决定性优势。”

In [ ]:
from sklearn.decomposition import PCA
import time

# --- 1. 离线预处理：1024 维压缩至 128 维 ---
print("⏳ 步骤 1: 正在将官方 BGE 特征进行‘脱水’压缩 (1024 -> 128)...")
# 我们在之前的步骤中已经拿到了 raw_embeddings [309, 1024]
pca_128 = PCA(n_components=128)
compressed_emb_v3 = pca_128.fit_transform(fixed_embeddings_v3.numpy())
fixed_embeddings_v3_light = torch.tensor(compressed_emb_v3, dtype=torch.float32)

# --- 2. 组装 128 维特征训练集 ---
X_list_v3_pc, y_list_v3_pc = [], []
for seq in corpus_seqs_v2_mapped:
    if len(seq) <= 4: continue
    for i in range(len(seq) - 4):
        # 输入维度: 4 * 128 = 512
        feat = fixed_embeddings_v3_light[seq[i : i+4]].view(-1)
        X_list_v3_pc.append(feat)
        y_list_v3_pc.append(seq[i+4])

X_train_v3_pc = torch.stack(X_list_v3_pc)
y_train_v3_pc = torch.tensor(y_list_v3_pc, dtype=torch.long)

# --- 3. 构造轻量化 FCN 模型 ---
class OptimizedFCN(nn.Module):
    def __init__(self, in_dim, vocab_size):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, vocab_size)
        )
    def forward(self, x): return self.net(x)

# 输入 512 维 -> 输出 309 个分类决策
model_v3_pc = OptimizedFCN(4 * 128, vocab_size_v2)
optimizer_v3_pc = optim.Adam(model_v3_pc.parameters(), lr=0.005) # 提高 LR

# --- 4. 极速训练 (2000 轮以内可跑出极低 Loss) ---
epochs_v3_pc = 1500
loss_hist_pc = []
start_pc = time.time()

for epoch in range(epochs_v3_pc):
    logits = model_v3_pc(X_train_v3_pc)
    loss = criterion(logits, y_train_v3_pc)
    
    optimizer_v3_pc.zero_grad(); loss.backward(); optimizer_v3_pc.step()
    loss_hist_pc.append(loss.item())
    
    if (epoch + 1) % 300 == 0:
        clear_output(wait=True)
        plt.figure(figsize=(10, 3.5))
        plt.plot(loss_hist_pc, color='teal', label='PCA Compressed Loss (512D)')
        plt.title(f"方案三 (压缩版) 训练曲线 - Epoch {epoch+1}")
        plt.xlabel("Epoch"); plt.ylabel("Loss"); plt.legend(); plt.show()

print(f"✅ 压缩版训练完成！总耗时: {time.time()-start_pc:.2f}s | 最终 Loss: {loss_hist_pc[-1]:.6f}")

# --- 5. 方案三 (压缩版) 文字接龙实测 ---
def generate_text_v3_optimized(prompt, model, pca_tool, max_len=20):
    model.eval()
    raw_ids = enc.encode(prompt)
    local_ids = [bpe2idx[tid] for tid in raw_ids if tid in bpe2idx]
    
    for _ in range(max_len):
        if len(local_ids) < 4:
            win_ids = [0]*(4-len(local_ids)) + local_ids
        else:
            win_ids = local_ids[-4:]
            
        # ⚡ 关键步骤：先在 128D 空间找特征 (离线查表)
        win_features = fixed_embeddings_v3_light[win_ids].view(1, -1)
        
        with torch.no_grad():
            output = model(win_features)
            next_idx = torch.argmax(output, dim=1).item()
            
        if idx2bpe[next_idx] == MAGIC_EOS_ID: break
        local_ids.append(next_idx)
        raw_ids.append(idx2bpe[next_idx])
        
    return enc.decode(raw_ids)

# --- 最终对比实测 ---
test_p = "神经网络是"
print(f"💡 输入: '{test_p}'")
print(f"🎯 方案三 (PCA 压缩优化) 结果: {generate_text_v3_optimized(test_p, model_v3_pc, pca_128)}")


为什么这次效果会更好？

- 特征密度大增：在 128 维空间，相似词的距离会由于剔除了“无关紧要的维度（噪声）”而靠得更近。模型不再需要在 4096 维的“无尽虚空”中寻找规律。
- 信号能量集中：PCA 提取的是数据中前 128 个最主要的“语义方向”。这相当于给模型配了一副“滤光镜”，只看最关键的特征，忽略次要干扰。
- 计算鲁棒性：参数量减少后，对于 258 行的小语料来说，模型更容易收敛到真正代表接龙逻辑的“局部最优解”，而不是在复杂的表面上迷路。

## 📉 4. 实验对比：滑动窗口大小的影响

模型能看到多少上下文（Context Window），直接决定了它能理解多复杂的关联细节。我们对比 2, 4, 8, 12 四种窗口大小下的表现。

在这个环节，我们将控制变量（使用相同的 FCN 架构和训练轮次），仅改变 窗口大小（即模型一次能“看”到多少个前面的 Token），验证它是如何影响预测准确率的。

实验设计说明 (Experimental Setup)：
- 模型标准化：所有实验均采用相同的 3 层全连接网络 (512x256)，以确保对比公平。
- 特征一致性：统一使用方案一中的“字符级分词 + 自建 Embedding”，排除外部 API 的干扰。
- 度量指标：记录每个窗口大小在 500 轮训练后的平均准确率 (Training Accuracy)。

In [ ]:
# --- 定义对比实验参数 ---
window_sizes_to_test = [2, 4, 8, 12] # 不同的视野范围
results_accuracy = []

print("🧪 正在启动窗口大小对比实验...")

for ws in window_sizes_to_test:
    # 步骤 A: 针对当前窗口大小重新构造数据集
    X_ws, y_ws = [], []
    for seq in corpus_seqs_v1:
        if len(seq) <= ws: continue
        for i in range(len(seq) - ws):
            X_ws.append(seq[i : i + ws])
            y_ws.append(seq[i + ws])
    
    X_ws_t = torch.tensor(X_ws, dtype=torch.long)
    y_ws_t = torch.tensor(y_ws, dtype=torch.long)
    
    # 步骤 B: 构建适配该窗口大小的模型 (3层 FCN)
    # 输入层规模 = 窗口大小 * Embedding 维度
    comp_model = nn.Sequential(
        nn.Embedding(vocab_size_v1, 64),
        nn.Flatten(),
        nn.Linear(ws * 64, 512), # 动态调整输入层
        nn.ReLU(),
        nn.Linear(512, 256),
        nn.ReLU(),
        nn.Linear(256, vocab_size_v1)
    )
    
    # 步骤 C: 快速训练 (统一 500 轮)
    opt = optim.Adam(comp_model.parameters(), lr=0.005)
    for epoch in range(500):
        logits = comp_model(X_ws_t)
        loss = criterion(logits, y_ws_t)
        opt.zero_grad(); loss.backward(); opt.step()
    
    # 步骤 D: 计算最终训练准确率
    with torch.no_grad():
        preds = torch.argmax(comp_model(X_ws_t), dim=1)
        acc = (preds == y_ws_t).float().mean().item()
        results_accuracy.append(acc)
        print(f"✅ 窗口大小 {ws:2d} 测试完成 | 最终准确率: {acc:.4f}")

# --- 绘制对比图表 ---
plt.figure(figsize=(10, 5))
plt.bar([str(w) for w in window_sizes_to_test], results_accuracy, color='slateblue', alpha=0.8)
plt.ylim(0, 1.1)
plt.title("不同滑动窗口 (Window Size) 对预测准确率的影响")
plt.xlabel("窗口大小 (Tokens)")
plt.ylabel("训练准确率 (Accuracy)")
# 在柱状图上方标注具体数值
for i, v in enumerate(results_accuracy):
    plt.text(i, v + 0.02, f"{v:.2%}", ha='center', fontweight='bold')
plt.show()

实验总结：为什么长窗口（长上下文）效果更好？
在上面的实验中，我们可以清晰地观察到：随着窗口大小的增加，模型的预测准确率呈上升趋势。 这是基于以下几个关键原因：

1. 消除“语义歧义” (Disambiguation)
短窗口 (如 2)：模型只能看到“是...”，在中文里“是”后面可以接万物（是机器、是人、是深度学习）。信息严重不足，模型只能处于“盲目猜测”状态。
长窗口 (如 12)：模型能看到“神经网络是通过反向传播来精细调整的...”，这时后面的 Token 概率会极大地向“参数”或“权重”收缩。
💡 结论：更多的上下文提供了更充足的约束条件。

2. 捕捉“长距离依赖” (Long-term Dependencies)
语言不是孤立的字串，而是一系列有逻辑的因果关系。有些关联发生在 10 个字甚至 100 个字之前。

窗口越长，模型越能理解句子的主语（Subject）是什么，从而在结尾处做出符合逻辑的收尾。
3. 为什么不无限增加窗口？
虽然窗口越长效果越好，但它也带来了两个主要的代价：

- 计算量爆炸：对于全连接网络，输入维度是 $WindowSize \times EmbeddingDim$。窗口翻倍，输入层参数也翻倍。
- 训练成本：对于 Transformer 架构，计算量通常随窗口大小呈 平方级 (Quadratic) 增长。这也就是为什么 GPT-4 或 Kimi 等标榜“长文本”能力的技术如此昂贵且难以实现的原因。

## 🎓 课堂总结

1. **大模型的秘密**：大模型的本质就是通过海量语料训练出来的、超大规模的“文字接龙”矩阵。
2. **分词与嵌入**：分词策略决定了编码效率，而 Embedding 为计算机提供了连接文本与语义的“数学桥梁”。
3. **窗口的重要性**：窗口就像模型的“视野”，视野越广，模型往往能表现出更强的逻辑自洽性。